In [1]:
from data_handler import DataHandlerModule
from model_handler import ModelHandlerModule

config = {
    "seed":           2,
    "data_name":      "ieee_cis",
    "raw_dir":        "./data",  # where CSVs live
    "sample":         10000,                                  # small sample to verify
    "multi_relation": True,
    "n_head":         [2, 2],
    "n_head_agg":     8,
    "feat_drop":      0,
    "attn_drop":      0,
    "train_ratio":    0.1,
    "test_ratio":     0.67,
    "emb_size":       [64, 64],
    "lr":             0.01,
    "weight_decay":   0.001,
    "epochs":         50,           # small for quick check
    "valid_epochs":   10,
    "batch_size":     1024,
    "patience":       20,
    "cuda_id":        0,
    "save_dir":       "./results/ieee_cis",
    "apply_gan":      True
}

In [2]:
data_handler  = DataHandlerModule(config)
model_handler = ModelHandlerModule(config, data_handler)
model_handler.train()

drag_model = model_handler.model
graph      = data_handler.dataset['graph']

Loading and preprocessing the dataset ieee_cis...
[IEEE-CIS] Loading from ./data ...... (GAN: True, GraphGAN: False)
  Sampled:   500000 rows  (fraud=17495, legit=482505)
  Running CTGAN to augment fraud samples...


/home/goofy/miniconda3/envs/pytorch310_debug/lib/python3.10/site-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name         Est # of Columns (CTGAN)
isFraud                      1
TransactionDT                11
TransactionAmt               11
ProductCD                    5
card1                        11
card2                        11
card3                        11
card4                        5
card5                        11
card6                        3
addr1                        11
addr2                        10
dist1                        11
C1                           11
C2                           11
C4                           11
C5                           11
C6                           11
C7                           11
C8                           11
C9                           11
C10                          11
C11                          11
C12                          11
C13                          11
C14 

/mnt/c/Users/Goofy/Documents/Projects/IFT-6759-AP-DRAG_Augmentation/model_handler.py:161: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(self

Test performance: - Epoch_Best: 49	- F1: 0.4121	- Recall: 0.7400	- Precision: 0.2856	- Accuracy: 0.8944	- AUC-ROC: 0.8964	- F1-macro: 0.6771	- Recall-macro: 0.8213	- AP: 0.6353	



In [3]:
from post_training.contrastive_drag import run_contrastive_pipeline
import copy

# Keep the original untouched
drag_model_original = drag_model

# Each call gets its own independent copy of the baseline weights
drag_model_direct_inference = run_contrastive_pipeline(
    copy.deepcopy(drag_model_original), graph, config,
    data_handler=data_handler, run_3a=True, run_3b=False
)


Device: cuda

PHASE 2 — Contrastive Fine-tuning
[Contrastive] Batch Size: 4096
[Contrastive] Encoder embedding dim: 64
  Epoch   1/50  Loss: 7.3766
  Epoch   2/50  Loss: 7.1837
  Epoch   3/50  Loss: 7.1497
  Epoch   4/50  Loss: 7.1321
  Epoch   5/50  Loss: 7.1199
  Epoch   6/50  Loss: 7.1112
  Epoch   7/50  Loss: 7.1053
  Epoch   8/50  Loss: 7.1007
  Epoch   9/50  Loss: 7.0974
  Epoch  10/50  Loss: 7.0943
  Epoch  11/50  Loss: 7.0920
  Epoch  12/50  Loss: 7.0896
  Epoch  13/50  Loss: 7.0881
  Epoch  14/50  Loss: 7.0866
  Epoch  15/50  Loss: 7.0855
  Epoch  16/50  Loss: 7.0844
  Epoch  17/50  Loss: 7.0834
  Epoch  18/50  Loss: 7.0828
  Epoch  19/50  Loss: 7.0820
  Epoch  20/50  Loss: 7.0813
  Epoch  21/50  Loss: 7.0810
  Epoch  22/50  Loss: 7.0803
  Epoch  23/50  Loss: 7.0802
  Epoch  24/50  Loss: 7.0796
  Epoch  25/50  Loss: 7.0791
  Epoch  26/50  Loss: 7.0787
  Epoch  27/50  Loss: 7.0785
  Epoch  28/50  Loss: 7.0781
  Epoch  29/50  Loss: 7.0779
  Epoch  30/50  Loss: 7.0776
  Epoch  31

In [4]:
drag_model_finetuned = run_contrastive_pipeline(
    copy.deepcopy(drag_model_original), graph, config,
    data_handler=data_handler, run_3a=False, run_3b=True
)

Device: cuda

PHASE 2 — Contrastive Fine-tuning
[Contrastive] Batch Size: 4096
[Contrastive] Encoder embedding dim: 64
  Epoch   1/50  Loss: 7.3787
  Epoch   2/50  Loss: 7.1859
  Epoch   3/50  Loss: 7.1519
  Epoch   4/50  Loss: 7.1352
  Epoch   5/50  Loss: 7.1242
  Epoch   6/50  Loss: 7.1160
  Epoch   7/50  Loss: 7.1097
  Epoch   8/50  Loss: 7.1047
  Epoch   9/50  Loss: 7.1004
  Epoch  10/50  Loss: 7.0974
  Epoch  11/50  Loss: 7.0946
  Epoch  12/50  Loss: 7.0928
  Epoch  13/50  Loss: 7.0906
  Epoch  14/50  Loss: 7.0891
  Epoch  15/50  Loss: 7.0876
  Epoch  16/50  Loss: 7.0865
  Epoch  17/50  Loss: 7.0853
  Epoch  18/50  Loss: 7.0844
  Epoch  19/50  Loss: 7.0832
  Epoch  20/50  Loss: 7.0825
  Epoch  21/50  Loss: 7.0816
  Epoch  22/50  Loss: 7.0806
  Epoch  23/50  Loss: 7.0800
  Epoch  24/50  Loss: 7.0793
  Epoch  25/50  Loss: 7.0784
  Epoch  26/50  Loss: 7.0781
  Epoch  27/50  Loss: 7.0778
  Epoch  28/50  Loss: 7.0775
  Epoch  29/50  Loss: 7.0770
  Epoch  30/50  Loss: 7.0769
  Epoch  31